### 1. Drive

In [ ]:
import os, cv2, random, glob, shutil
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from google.colab import drive

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

drive.mount('/content/drive')

CHUNK_DIR = '/content/drive/MyDrive/img_celeba.zip'
BBOX_FILE_PATH = '/content/drive/MyDrive/list_bbox_celeba.txt'
EXTRACT_DIR = '/content/celeba'

if os.path.exists(EXTRACT_DIR):
    shutil.rmtree(EXTRACT_DIR)

os.makedirs(EXTRACT_DIR, exist_ok=True)

TEMP_ZIP_PATH = '/tmp/img_celeba.zip'
!cp "{CHUNK_DIR}" "{TEMP_ZIP_PATH}"

!unzip -q {TEMP_ZIP_PATH} -d {EXTRACT_DIR}

IMG_DIR = os.path.join(EXTRACT_DIR, 'img_align_celeba')

Mounted at /content/drive


In [ ]:
import os

print(f"{EXTRACT_DIR}:")
!ls {EXTRACT_DIR}

found_img_dir = None
for root, dirs, files in os.walk(EXTRACT_DIR):
    if any(f.endswith('.jpg') for f in files):
        found_img_dir = root
        break

if found_img_dir:
    IMG_DIR = found_img_dir
    print(f"Số lượng ảnh: {len([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')])}")
else:
    print("\none")

/content/celeba:
img_celeba
Số lượng ảnh: 202599


### 2. Xử lý Dữ liệu & YOLO Grid Mapping

In [ ]:
def parse_celeba(bbox_file, min_size=80):
    data = []
    if not os.path.exists(bbox_file):
        print(f"Không tìm thấy file {bbox_file}")
        return []
    with open(bbox_file, 'r') as f:
        lines = f.readlines()[2:]

    for line in lines:
        parts = line.strip().split()
        name, x, y, w, h = parts[0], int(parts[1]), int(parts[2]), int(parts[3]), int(parts[4])
        if w >= min_size and h >= min_size:
            data.append((name, [[x, y, w, h]]))
    return data

all_pos_data = parse_celeba(BBOX_FILE_PATH)
all_imgs = [f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')]

if len(all_imgs) == 0:
    print(f"Không tìm thấy ảnh {IMG_DIR}.")
    neg_names = []
else:
    pos_names = set([d[0] for d in all_pos_data])
    available_negs = [n for n in all_imgs if n not in pos_names]
    sample_size = min(2000, len(available_negs))
    neg_names = random.sample(available_negs, sample_size) if sample_size > 0 else []

final_data = all_pos_data + [(n, []) for n in neg_names]
random.shuffle(final_data)

if len(final_data) > 0:
    split = int(0.8 * len(final_data))
    train_data, val_data = final_data[:split], final_data[split:]
    print(f"Tổng cộng: {len(final_data)} mẫu.")
    print(f"Train: {len(train_data)}, Val: {len(val_data)}")
else:
    print("none")

Tổng cộng: 192183 mẫu.
Train: 153746, Val: 38437


In [ ]:
import pandas as pd

with open(BBOX_FILE_PATH, 'r') as f:
    total_labels = len(f.readlines()) - 2

images_on_disk = set([f for f in os.listdir(IMG_DIR) if f.endswith('.jpg')])
labels_in_file = set([d[0] for d in all_pos_data])

matching_images = images_on_disk.intersection(labels_in_file)

print(f"Tổng nhãn: {total_labels}")
print(f"Tổng ảnh: {len(images_on_disk)}")
print(f"Face >= 80x80: {len(matching_images)}")

Tổng nhãn: 202599
Tổng ảnh: 202599
Face >= 80x80: 190183


In [ ]:
class CelebADataset(Dataset):
    def __init__(self, data, img_dir, S=13, img_size=416, augment=True):
        self.data = data
        self.img_dir = img_dir
        self.S = S
        self.img_size = img_size
        self.augment = augment

    def __len__(self): return len(self.data)

    def __getitem__(self, idx):
        img_name, boxes = self.data[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h_orig, w_orig = img.shape[:2]

        if self.augment:
            if random.random() > 0.5:
                img = cv2.flip(img, 1)
                boxes = [[w_orig - x - w, y, w, h] for x, y, w, h in boxes]
            alpha = random.uniform(0.8, 1.2)
            beta = random.uniform(-20, 20)
            img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)
            if random.random() > 0.8:
                img = cv2.GaussianBlur(img, (5, 5), 0)

        img = cv2.resize(img, (self.img_size, self.img_size))
        img_tensor = torch.from_numpy(img).permute(2, 0, 1).float() / 255.0

        target = torch.zeros((self.S, self.S, 5))
        for x, y, bw, bh in boxes:
            cx, cy = (x + bw/2) / w_orig, (y + bh/2) / h_orig
            nw, nh = bw / w_orig, bh / h_orig

            i, j = int(self.S * cy), int(self.S * cx)
            if 0 <= i < self.S and 0 <= j < self.S:
                x_cell = self.S * cx - j
                y_cell = self.S * cy - i
                target[i, j] = torch.tensor([x_cell, y_cell, nw, nh, 1.0])

        return img_tensor, target




### 3. Model & Loss Function

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

def bbox_iou_giou(box1, box2):
    b1_x1, b1_y1, b1_x2, b1_y2 = box1[..., 0], box1[..., 1], box1[..., 2], box1[..., 3]
    b2_x1, b2_y1, b2_x2, b2_y2 = box2[..., 0], box2[..., 1], box2[..., 2], box2[..., 3]

    inter_x1 = torch.max(b1_x1, b2_x1)
    inter_y1 = torch.max(b1_y1, b2_y1)
    inter_x2 = torch.min(b1_x2, b2_x2)
    inter_y2 = torch.min(b1_y2, b2_y2)

    inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)

    b1_area = (b1_x2 - b1_x1) * (b1_y2 - b1_y1)
    b2_area = (b2_x2 - b2_x1) * (b2_y2 - b2_y1)
    union_area = b1_area + b2_area - inter_area + 1e-6

    iou = inter_area / union_area

    c_x1 = torch.min(b1_x1, b2_x1)
    c_y1 = torch.min(b1_y1, b2_y1)
    c_x2 = torch.max(b1_x2, b2_x2)
    c_y2 = torch.max(b1_y2, b2_y2)

    c_area = (c_x2 - c_x1) * (c_y2 - c_y1) + 1e-6

    giou = iou - (c_area - union_area) / c_area

    return giou

class SimpleFaceDetector(nn.Module):
    def __init__(self, S=13):
        super().__init__()
        def block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1, bias=False),
                nn.BatchNorm2d(out_c),
                nn.LeakyReLU(0.1)
            )

        self.model = nn.Sequential(
            block(3, 32),
            block(32, 32), nn.MaxPool2d(2),
            block(32, 64),
            block(64, 64), nn.MaxPool2d(2),
            block(64, 128),
            block(128, 128), nn.MaxPool2d(2),
            block(128, 256),
            block(256, 256), nn.MaxPool2d(2),
            block(256, 512),
            block(512, 512), nn.MaxPool2d(2),
            block(512, 256),
        )
        self.head = nn.Conv2d(256, 5, kernel_size=1)

    def forward(self, x):
        x = self.model(x)
        x = self.head(x)
        x = x.permute(0, 2, 3, 1)

        boxes = torch.sigmoid(x[..., :4])
        conf = x[..., 4:5]

        return torch.cat([boxes, conf], dim=-1)

def compute_loss(pred, target, S=13, lambda_giou=1.0, lambda_obj_conf=1.0, lambda_noobj_conf=0.5):
    obj_mask = target[..., 4] == 1
    noobj_mask = target[..., 4] == 0

    loss_giou = torch.tensor(0.0, device=pred.device)
    loss_obj_conf = torch.tensor(0.0, device=pred.device)

    if obj_mask.any():
        pred_obj = pred[obj_mask]
        target_obj = target[obj_mask]

        _, i_indices, j_indices = torch.where(obj_mask)

        pred_boxes_img_abs = torch.stack([
            (pred_obj[:, 0] + j_indices.float()) / S,
            (pred_obj[:, 1] + i_indices.float()) / S,
            pred_obj[:, 2],
            pred_obj[:, 3]
        ], dim=-1)

        target_boxes_img_abs = torch.stack([
            (target_obj[:, 0] + j_indices.float()) / S,
            (target_obj[:, 1] + i_indices.float()) / S,
            target_obj[:, 2],
            target_obj[:, 3]
        ], dim=-1)

        def box_cxcywh_to_xyxy(boxes):
            x1 = boxes[..., 0] - boxes[..., 2] / 2
            y1 = boxes[..., 1] - boxes[..., 3] / 2
            x2 = boxes[..., 0] + boxes[..., 2] / 2
            y2 = boxes[..., 1] + boxes[..., 3] / 2
            return torch.stack([x1, y1, x2, y2], dim=-1)

        pred_boxes_xyxy = box_cxcywh_to_xyxy(pred_boxes_img_abs)
        target_boxes_xyxy = box_cxcywh_to_xyxy(target_boxes_img_abs)

        giou_values = bbox_iou_giou(pred_boxes_xyxy, target_boxes_xyxy)
        loss_giou = torch.sum(1 - giou_values)

        loss_obj_conf = F.binary_cross_entropy_with_logits(pred_obj[:, 4], target_obj[:, 4], reduction='sum')

    loss_noobj_conf = F.binary_cross_entropy_with_logits(pred[noobj_mask][:, 4], target[noobj_mask][:, 4], reduction='sum')

    total_loss = lambda_giou * loss_giou + lambda_obj_conf * loss_obj_conf + lambda_noobj_conf * loss_noobj_conf
    return total_loss



### 4. Training Loop



In [ ]:
import os, cv2, torch, random, glob, shutil
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm
from google.colab import drive
import torchvision.ops as ops

def box_cxcywh_to_xyxy(boxes):
    x1 = boxes[..., 0] - boxes[..., 2] / 2
    y1 = boxes[..., 1] - boxes[..., 3] / 2
    x2 = boxes[..., 0] + boxes[..., 2] / 2
    y2 = boxes[..., 1] + boxes[..., 3] / 2
    return torch.stack([x1, y1, x2, y2], dim=-1)

def batch_iou(boxes1, boxes2):

    if boxes1.numel() == 0 or boxes2.numel() == 0:
        return torch.empty((boxes1.shape[0] if boxes1.numel() > 0 else 0,
                            boxes2.shape[0] if boxes2.numel() > 0 else 0),
                           device=boxes1.device if boxes1.numel() > 0 else boxes2.device)


    boxes1_exp = boxes1.unsqueeze(1)
    boxes2_exp = boxes2.unsqueeze(0)

    inter_x1 = torch.max(boxes1_exp[..., 0], boxes2_exp[..., 0])
    inter_y1 = torch.max(boxes1_exp[..., 1], boxes2_exp[..., 1])
    inter_x2 = torch.min(boxes1_exp[..., 2], boxes2_exp[..., 2])
    inter_y2 = torch.min(boxes1_exp[..., 3], boxes2_exp[..., 3])

    inter_area = torch.clamp(inter_x2 - inter_x1, min=0) * torch.clamp(inter_y2 - inter_y1, min=0)

    area1 = (boxes1_exp[..., 2] - boxes1_exp[..., 0]) * (boxes1_exp[..., 3] - boxes1_exp[..., 1])
    area2 = (boxes2_exp[..., 2] - boxes2_exp[..., 0]) * (boxes2_exp[..., 3] - boxes2_exp[..., 1])

    union_area = area1 + area2 - inter_area + 1e-6

    return inter_area / union_area


def evaluate(model, loader, device, S=13, conf_threshold=0.5, iou_threshold=0.5, nms_iou_threshold=0.3):
    model.eval()
    tp_total, fp_total, fn_total = 0, 0, 0
    total_tp_iou_sum = 0.0

    j_coords = torch.arange(S, device=device, dtype=torch.float32)
    i_coords = torch.arange(S, device=device, dtype=torch.float32)

    grid_indices = torch.stack(torch.meshgrid(j_coords, i_coords, indexing='xy'), dim=-1).view(-1, 2)

    with torch.no_grad():
        for imgs, targets in loader:
            imgs = imgs.to(device)
            targets = targets.to(device)
            preds = model(imgs)

            for b in range(preds.shape[0]):
                preds_b = preds[b]
                targets_b = targets[b]

                preds_flat = preds_b.view(-1, 5)

                conf_logits = preds_flat[:, 4]
                conf_scores = torch.sigmoid(conf_logits)

                conf_mask = conf_scores > conf_threshold
                filtered_pred_boxes_raw = preds_flat[conf_mask, :4]
                filtered_pred_scores = conf_scores[conf_mask]


                confident_grid_indices = grid_indices[conf_mask]

                pred_boxes_xyxy = torch.empty((0, 4), device=device)
                if len(filtered_pred_boxes_raw) > 0:
                    cx_cell, cy_cell, nw, nh = filtered_pred_boxes_raw.split(1, dim=1)


                    cx_img = (cx_cell.squeeze(-1) + confident_grid_indices[:, 0]) / S
                    cy_img = (cy_cell.squeeze(-1) + confident_grid_indices[:, 1]) / S

                    pred_boxes_cxcywh_img = torch.stack([cx_img, cy_img, nw.squeeze(-1), nh.squeeze(-1)], dim=-1)
                    pred_boxes_xyxy = box_cxcywh_to_xyxy(pred_boxes_cxcywh_img)

                nms_boxes = torch.empty((0, 4), device=device)
                nms_scores = torch.empty((0,), device=device)
                if len(pred_boxes_xyxy) > 0:

                    keep_indices = ops.nms(pred_boxes_xyxy, filtered_pred_scores, nms_iou_threshold)
                    nms_boxes = pred_boxes_xyxy[keep_indices]
                    nms_scores = filtered_pred_scores[keep_indices]


                targets_flat = targets_b.view(-1, 5)
                obj_mask_gt = (targets_flat[:, 4] == 1)

                filtered_gt_boxes_raw = targets_flat[obj_mask_gt, :4]
                gt_grid_indices = grid_indices[obj_mask_gt]

                gt_boxes_xyxy = torch.empty((0, 4), device=device)
                if len(filtered_gt_boxes_raw) > 0:
                    cx_cell_gt, cy_cell_gt, nw_gt, nh_gt = filtered_gt_boxes_raw.split(1, dim=1)

                    gt_grid_indices = gt_grid_indices.to(device)

                    cx_img_gt = (cx_cell_gt.squeeze(-1) + gt_grid_indices[:, 0]) / S
                    cy_img_gt = (cy_cell_gt.squeeze(-1) + gt_grid_indices[:, 1]) / S

                    gt_boxes_cxcywh_img = torch.stack([cx_img_gt, cy_img_gt, nw_gt.squeeze(-1), nh_gt.squeeze(-1)], dim=-1)
                    gt_boxes_xyxy = box_cxcywh_to_xyxy(gt_boxes_cxcywh_img)

                tp_batch, fp_batch, fn_batch = 0, 0, 0

                num_preds = len(nms_boxes)
                num_gts = len(gt_boxes_xyxy)

                if num_preds > 0 and num_gts > 0:
                    iou_matrix = batch_iou(nms_boxes, gt_boxes_xyxy)

                    max_ious_per_pred, best_gt_indices_per_pred = iou_matrix.max(dim=1)

                    matched_gts_mask = torch.zeros(num_gts, dtype=torch.bool, device=device)

                    sorted_pred_indices = torch.argsort(nms_scores, descending=True)

                    for p_idx_in_sorted_preds in sorted_pred_indices:
                        current_pred_iou = max_ious_per_pred[p_idx_in_sorted_preds]
                        best_gt_for_current_pred = best_gt_indices_per_pred[p_idx_in_sorted_preds]

                        if current_pred_iou >= iou_threshold and not matched_gts_mask[best_gt_for_current_pred]:
                            tp_batch += 1
                            total_tp_iou_sum += current_pred_iou.item()
                            matched_gts_mask[best_gt_for_current_pred] = True
                        else:
                            fp_batch += 1

                    fn_batch = num_gts - matched_gts_mask.sum().item()

                elif num_preds > 0:
                    fp_batch = num_preds
                elif num_gts > 0:
                    fn_batch = num_gts

                tp_total += tp_batch
                fp_total += fp_batch
                fn_total += fn_batch

    prec = tp_total / (tp_total + fp_total + 1e-6)
    rec = tp_total / (tp_total + fn_total + 1e-6)
    f1 = 2 * (prec * rec) / (prec + rec + 1e-6)

    mean_iou = total_tp_iou_sum / tp_total if tp_total > 0 else 0.0

    return prec, rec, f1, mean_iou

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = SimpleFaceDetector().to(device)
model = torch.compile(model)
optimizer = optim.Adam(model.parameters(), lr=1e-4)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.1,
    patience=2,
    # verbose=True
)

num_workers = os.cpu_count() // 2 if device == 'cuda' else 0
pin_memory = True if device == 'cuda' else False

train_loader = DataLoader(
    CelebADataset(train_data, IMG_DIR),
    batch_size=256,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=pin_memory
)
val_loader = DataLoader(
    CelebADataset(val_data, IMG_DIR, augment=False),
    batch_size=256,
    num_workers=num_workers,
    pin_memory=pin_memory
)

train_losses = []
val_precisions = []
val_recalls = []
val_f1_scores = []
val_mean_ious = []

log_file_path = '/content/training_log.txt'
with open(log_file_path, 'w') as f:
    f.write('Epoch\tTrain Loss\tVal Precision\tVal Recall\tVal F1-Score\tVal mIoU\tLearning Rate\n')

scaler = torch.amp.GradScaler('cuda', enabled=(device == 'cuda'))

CHECKPOINT_DIR = '/content/drive/MyDrive/face_detector_checkpoints'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

start_epoch = 0

checkpoint_files = sorted(glob.glob(os.path.join(CHECKPOINT_DIR, 'face_detector_epoch_*.pth')))
if checkpoint_files:
    latest_checkpoint = checkpoint_files[-1]
    print(f"Found latest checkpoint: {latest_checkpoint}")
    checkpoint = torch.load(latest_checkpoint)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if 'scheduler_state_dict' in checkpoint:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
        print("Scheduler state loaded from checkpoint.")
    start_epoch = checkpoint['epoch']
    print(f"Resuming training from epoch {start_epoch + 1}")
else:
    print("Start from epoch 1")



NUM_EPOCHS = 20
for epoch in range(start_epoch, NUM_EPOCHS):
    model.train()
    total_loss = 0
    LAMBDA_GIOU = 5.0
    LAMBDA_OBJ_CONF = 1.0
    LAMBDA_NOOBJ_CONF = 0.5

    torch.cuda.empty_cache()

    for imgs, targets in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
        imgs, targets = imgs.to(device), targets.to(device)
        optimizer.zero_grad()

        with torch.amp.autocast('cuda', enabled=(device == 'cuda')):

            loss = compute_loss(model(imgs), targets, S=13,
                                lambda_giou=LAMBDA_GIOU,
                                lambda_obj_conf=LAMBDA_OBJ_CONF,
                                lambda_noobj_conf=LAMBDA_NOOBJ_CONF)



        scaler.scale(loss).backward()

        scaler.unscale_(optimizer)

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()

    avg_train_loss = total_loss / len(train_loader)

    torch.cuda.empty_cache()
    p, r, f1, mean_iou = evaluate(model, val_loader, device, S=13)

    scheduler.step(f1)

    train_losses.append(avg_train_loss)
    val_precisions.append(p)
    val_recalls.append(r)
    val_f1_scores.append(f1)
    val_mean_ious.append(mean_iou)

    current_lr = optimizer.param_groups[0]['lr']

    print(f"Epoch {epoch+1} | Train Loss: {avg_train_loss:.4f} | Val P: {p:.2f} R: {r:.2f} F1: {f1:.2f} | mIoU: {mean_iou:.2f} | LR: {current_lr:.6f}")

    with open(log_file_path, 'a') as f:
        f.write(f'{epoch+1}\t{avg_train_loss:.4f}\t{p:.2f}\t{r:.2f}\t{f1:.2f}\t{mean_iou:.2f}\t{current_lr:.6f}\n')

    checkpoint_path = os.path.join(CHECKPOINT_DIR, f'face_detector_epoch_{epoch+1}.pth')
    torch.save({
        'epoch': epoch + 1,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'train_loss': avg_train_loss,
        'val_precision': p,
        'val_recall': r,
        'val_f1': f1,
        'val_mean_iou': mean_iou,
    }, checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}")

    shutil.copy(log_file_path, CHECKPOINT_DIR)
    print(f"Training log copied to {CHECKPOINT_DIR}")

print(f"final saved to {CHECKPOINT_DIR}")

Checkpoints will be saved to: /content/drive/MyDrive/face_detector_checkpoints
No existing checkpoints found. Starting training from epoch 1.


Epoch 1:   0%|          | 0/601 [00:00<?, ?it/s]

Epoch 1 | Train Loss: 2008.4212 | Val P: 0.98 R: 0.93 F1: 0.96 | mIoU: 0.87 | LR: 0.000100
Checkpoint saved to /content/drive/MyDrive/face_detector_checkpoints/face_detector_epoch_1.pth
Training log copied to /content/drive/MyDrive/face_detector_checkpoints


Epoch 2:   0%|          | 0/601 [00:00<?, ?it/s]

Epoch 2 | Train Loss: 398.1682 | Val P: 0.99 R: 0.95 F1: 0.97 | mIoU: 0.89 | LR: 0.000100
Checkpoint saved to /content/drive/MyDrive/face_detector_checkpoints/face_detector_epoch_2.pth
Training log copied to /content/drive/MyDrive/face_detector_checkpoints


...

Epoch 8:   0%|          | 0/601 [00:00<?, ?it/s]

Epoch 8 | Train Loss: 269.4658 | Val P: 0.99 R: 0.98 F1: 0.99 | mIoU: 0.90 | LR: 0.000100
Checkpoint saved to /content/drive/MyDrive/face_detector_checkpoints/face_detector_epoch_8.pth
Training log copied to /content/drive/MyDrive/face_detector_checkpoints


Epoch 9:   0%|          | 0/601 [00:00<?, ?it/s]

Epoch 9 | Train Loss: 264.2353 | Val P: 0.99 R: 0.98 F1: 0.98 | mIoU: 0.90 | LR: 0.000100
Checkpoint saved to /content/drive/MyDrive/face_detector_checkpoints/face_detector_epoch_9.pth
Training log copied to /content/drive/MyDrive/face_detector_checkpoints


Epoch 10:   0%|          | 0/601 [00:00<?, ?it/s]